### Pyspark - RDD Concepts, GroupBy And Aggregate Functions

Jay Urbain, PhD   
3/18/2024


PySpark documentation:   
https://spark.apache.org/docs/latest/api/python/reference/index.html  

Answer the TODO items.



![](spark-cluster-overview.webp)

In [1]:
# import os
from pyspark.sql import SparkSession
import pandas as pd

In [2]:
spark = SparkSession.builder \
    .appName("Intro") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/01 19:23:53 WARN Utils: Your hostname, Alpha, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/01 19:23:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/01 19:23:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/01 19:23:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/01 19:23:54 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


4.1.1


In order to create an RDD, first, you need to create a SparkSession which is an entry point to the PySpark application. SparkSession can be created using a builder() or newSession() methods of the SparkSession.

Spark session internally creates a sparkContext variable of SparkContext. You can create multiple SparkSession objects but only one SparkContext per JVM. In case if you want to create another new SparkContext you should stop existing Sparkcontext (using stop()) before creating a new one.

#### PySpark RDD – Resilient Distributed Dataset

PySpark RDD (Resilient Distributed Dataset) is a fundamental data structure of PySpark that is fault-tolerant, immutable distributed collections of objects, which means once you create an RDD you cannot change it. Each dataset in RDD is divided into logical partitions, which can be computed on different nodes of the cluster.

In order to create an RDD, we need to create a SparkSession which is an entry point to the PySpark application. SparkSession can be created using a builder() or newSession() methods of the SparkSession.

Spark session internally creates a sparkContext variable of SparkContext. We can create multiple SparkSession objects but only one SparkContext per JVM. In case if you want to create another new SparkContext you should stop existing Sparkcontext (using stop()) before creating a new one.

If you have tabular data, you should just use a DataFrame. Going down to the RDD layer is helpful for unstructured data, for example, text, images, and signal data.


In [3]:
# Create RDD from list using parallelize

dataList = [("Java", 2000), ("Python", 10000), ("Scala", 3000)]
rdd=spark.sparkContext.parallelize(dataList)

In [4]:
# Create an RDD from external data source

dataList = [("Java", 2000), ("Python", 10000), ("Scala", 3000)]
rdd2=spark.sparkContext.textFile('test3.csv')

#### RDD Operations

On PySpark RDD, you can perform two kinds of operations.

RDD transformations – Transformations are lazy operations. When you run a transformation(for example update), instead of updating a current RDD, these operations return another RDD.

RDD actions – operations that trigger computation and return RDD values to the driver.


#### RDD Transformations

Transformations on Spark RDD returns another RDD and transformations are lazy meaning they don’t execute until you call an action on RDD. Some transformations on RDD’s are flatMap(), map(), reduceByKey(), filter(), sortByKey() and return new RDD instead of updating the current.

#### PySpark DataFrame

DataFrame is a distributed collection of data organized into named columns. It is conceptually equivalent to a table in a relational database or a data frame in R/Python, but with  optimizations under the hood. DataFrames can be constructed from a wide array of sources such as structured data files, tables in Hive, external databases, or existing RDDs.


#### TODO: Is PySpark faster than pandas? Explain your thinking.**
    
    

    
    

Create a dataframe

In [5]:
df_pyspark=spark.read.csv('test3.csv',header=True,inferSchema=True)

In [6]:
df_pyspark.show()

+---------+------------+------+
|     Name| Departments|salary|
+---------+------------+------+
|    Krish|Data Science| 10000|
|    Krish|         IOT|  5000|
|   Mahesh|    Big Data|  4000|
|    Krish|    Big Data|  4000|
|   Mahesh|Data Science|  3000|
|Sudhanshu|Data Science| 20000|
|Sudhanshu|         IOT| 10000|
|Sudhanshu|    Big Data|  5000|
|    Sunny|Data Science| 10000|
|    Sunny|    Big Data|  2000|
+---------+------------+------+



In [7]:
df_pyspark.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Departments: string (nullable = true)
 |-- salary: integer (nullable = true)



PySpark SQL is one of the most used PySpark modules which is used for processing structured columnar data format. Once you have a DataFrame created, you can interact with the data by using SQL syntax.

In other words, Spark SQL brings native SQL queries on Spark meaning you can run traditional ANSI SQL’s on Spark Dataframe, in the later section of this PySpark SQL tutorial, you will learn in detail using SQL select, where, group by, join, union e.t.c

In order to use SQL, first, create a temporary table on DataFrame using createOrReplaceTempView() function. Once created, this table can be accessed throughout the SparkSession using sql() and it will be dropped along with your SparkContext termination.

Use sql() method of the SparkSession object to run the query and this method returns a new DataFrame.

Reference:

https://spark.apache.org/docs/3.1.2/api/python/reference/pyspark.sql.html

In [8]:
df_pyspark.createOrReplaceTempView("PEOPLE")
df1=spark.sql("select Name, Departments, Salary as salary from PEOPLE")
df1.show()

+---------+------------+------+
|     Name| Departments|salary|
+---------+------------+------+
|    Krish|Data Science| 10000|
|    Krish|         IOT|  5000|
|   Mahesh|    Big Data|  4000|
|    Krish|    Big Data|  4000|
|   Mahesh|Data Science|  3000|
|Sudhanshu|Data Science| 20000|
|Sudhanshu|         IOT| 10000|
|Sudhanshu|    Big Data|  5000|
|    Sunny|Data Science| 10000|
|    Sunny|    Big Data|  2000|
+---------+------------+------+



In [9]:
## Groupby

# Grouped to find the maximum salary
df_pyspark.groupBy('Name').sum().show()

+---------+-----------+
|     Name|sum(salary)|
+---------+-----------+
|Sudhanshu|      35000|
|    Sunny|      12000|
|    Krish|      19000|
|   Mahesh|       7000|
+---------+-----------+



In [10]:
#### TODO: Perform the operation above using PySpark SQL

# Run the equivalent SQL query
df_sum = spark.sql("""
    SELECT Name, SUM(Salary) AS sum_salary
    FROM PEOPLE
    GROUP BY Name
""")

# Display the results
df_sum.show()


+---------+----------+
|     Name|sum_salary|
+---------+----------+
|Sudhanshu|     35000|
|    Sunny|     12000|
|    Krish|     19000|
|   Mahesh|      7000|
+---------+----------+



In [11]:
df_pyspark.groupBy('Name').avg().show()

+---------+------------------+
|     Name|       avg(salary)|
+---------+------------------+
|Sudhanshu|11666.666666666666|
|    Sunny|            6000.0|
|    Krish| 6333.333333333333|
|   Mahesh|            3500.0|
+---------+------------------+



In [12]:
#### TODO: Use PySpark dataframe to group by departmebnnts and compute sum for each department

# Group by Departments and compute sum of salary
df_sum = df_pyspark.groupBy("Departments").sum("salary").withColumnRenamed("sum(salary)", "Total_Salary")

# Show results
df_sum.show()


+------------+------------+
| Departments|Total_Salary|
+------------+------------+
|         IOT|       15000|
|    Big Data|       15000|
|Data Science|       43000|
+------------+------------+



In [13]:
#### TODO: Use PySpark SQL to group by departmebnnts and compute sum for each department
# Use Spark SQL to group by Departments and sum salaries
 
df_pyspark.createOrReplaceTempView("EMPLOYEES")
# Group by Departments and sum salaries
df_sum_sql = spark.sql("""
    SELECT Departments, SUM(salary) AS Total_Salary
    FROM EMPLOYEES
    GROUP BY Departments
""")

# Show the results
df_sum_sql.show()


+------------+------------+
| Departments|Total_Salary|
+------------+------------+
|         IOT|       15000|
|    Big Data|       15000|
|Data Science|       43000|
+------------+------------+



In [14]:
#### TODO: Use PySpark dataframe to group and compute count for each department

# Group by Departments and compute count
df_count = df_pyspark.groupBy("Departments").count()

# Show results
df_count.show()


+------------+-----+
| Departments|count|
+------------+-----+
|         IOT|    2|
|    Big Data|    4|
|Data Science|    4|
+------------+-----+



Another method

In [15]:
df_pyspark.agg({'Salary':'sum'}).show()

+-----------+
|sum(Salary)|
+-----------+
|      73000|
+-----------+

